# Apex PR2012 autograd callback review
Pinned base becbb77cea4cb54f2929f7c938a0a6f7dd1fdc39 and head 87584037722e3f3e8753b29c5159bbe8bbec5dbc. Exact setup_context callbacks with identity probe ops; no Apex CUDA extension execution or inference-leak result claimed.


In [ ]:
import pathlib, subprocess, json
pathlib.Path("base.py").write_text("import importlib\nimport numbers\n\nimport torch\nfrom torch.nn.parameter import Parameter\nfrom torch.nn import init\nfrom torch.nn import functional as F\nfrom typing import List, Tuple\n\nfrom apex._autocast_utils import _cast_if_autocast_enabled\n\nglobal fused_layer_norm_cuda\nfused_layer_norm_cuda = None\n\n\n# PyTorch supports `torch.library.custom_op` since 2.4.0.\ndef supports_custom_op() -> bool:\n    return hasattr(torch.library, \"custom_op\")\n\n\n# Reference implementation from Huggingface\ndef manual_rms_norm(input, normalized_shape, weight, eps):\n    # layer norm should always be calculated in float32\n    dims = tuple(i for i in range(-1, -len(normalized_shape) - 1, -1))\n    variance = input.to(torch.float32).pow(2).mean(dims, keepdim=True)\n    input = input * torch.rsqrt(variance + eps)\n\n    if weight is None:\n        return input\n\n    # convert into half-precision if necessary\n    if weight.dtype in [torch.float16, torch.bfloat16]:\n        input = input.to(weight.dtype)\n\n    return weight * input\n\n\nclass FusedLayerNormAffineFunction(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, input, weight, bias, normalized_shape, eps, memory_efficient=False):\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n        input_ = input.contiguous()\n        weight_ = weight.contiguous()\n        bias_ = bias.contiguous()\n        output, mean, invvar = fused_layer_norm_cuda.forward_affine(\n            input_, ctx.normalized_shape, weight_, bias_, ctx.eps\n        )\n        if ctx.memory_efficient:\n            ctx.save_for_backward(output, weight_, bias_, None, invvar)\n        else:\n            ctx.save_for_backward(input_, weight_, bias_, mean, invvar)\n        return output\n\n    @staticmethod\n    def backward(ctx, grad_output):\n        input_or_output, weight_, bias_, mean, invvar = ctx.saved_tensors\n        grad_input = grad_weight = grad_bias = None\n        grad_input, grad_weight, grad_bias = fused_layer_norm_cuda.backward_affine(\n            grad_output.contiguous(),\n            mean,\n            invvar,\n            input_or_output,\n            ctx.normalized_shape,\n            weight_,\n            bias_,\n            ctx.eps,\n            ctx.memory_efficient,\n        )\n        return grad_input, grad_weight, grad_bias, None, None, None\n\n\nif supports_custom_op():\n\n    @torch.library.custom_op(\"apex::fused_layer_norm_affine_fwd\", mutates_args=())\n    def fused_layer_norm_affine_fwd(\n        input: torch.Tensor,\n        weight: torch.Tensor,\n        bias: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n\n        input_ = input.contiguous()\n        weight_ = weight.contiguous()\n        bias_ = bias.contiguous()\n        output, mean, invvar = fused_layer_norm_cuda.forward_affine(\n            input_, normalized_shape, weight_, bias_, eps\n        )\n        return output, mean, invvar\n\n    @fused_layer_norm_affine_fwd.register_fake\n    def fused_layer_norm_affine_fwd_fake(\n        input: torch.Tensor,\n        weight: torch.Tensor,\n        bias: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        input = input.contiguous()\n        weight = weight.contiguous()\n        bias = bias.contiguous()\n        idiff = input.ndim - len(normalized_shape)\n        n = 1\n        for i in range(idiff):\n            n *= input.shape[i]\n        if input.dtype in [torch.float16, torch.bfloat16]:\n            dtype = torch.float32\n        else:\n            dtype = input.dtype\n        mean = torch.empty([n], dtype=dtype, device=input.device)\n        invvar = torch.empty_like(mean)\n        return torch.empty_like(input), mean, invvar\n\n    @torch.library.custom_op(\"apex::fused_layer_norm_affine_bwd\", mutates_args=())\n    def fused_layer_norm_affine_bwd(\n        grad_output: torch.Tensor,\n        mean: torch.Tensor,\n        invvar: torch.Tensor,\n        input_or_output: torch.Tensor,\n        normalized_shape: List[int],\n        weight: torch.Tensor,\n        bias: torch.Tensor,\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        grad_input, grad_weight, grad_bias = fused_layer_norm_cuda.backward_affine(\n            grad_output.contiguous(),\n            mean,\n            invvar,\n            input_or_output,\n            normalized_shape,\n            weight,\n            bias,\n            eps,\n            memory_efficient,\n        )\n        return grad_input, grad_weight, grad_bias\n\n    @fused_layer_norm_affine_bwd.register_fake\n    def fused_layer_norm_affine_bwd_fake(\n        grad_output: torch.Tensor,\n        mean: torch.Tensor,\n        invvar: torch.Tensor,\n        input_or_output: torch.Tensor,\n        normalized_shape: List[int],\n        weight: torch.Tensor,\n        bias: torch.Tensor,\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        grad_input = torch.empty_like(input_or_output)\n        grad_weight = torch.empty_like(weight)\n        grad_bias = torch.empty_like(bias)\n        return grad_input, grad_weight, grad_bias\n\n    def _fused_layer_norm_affine_backward(ctx, grad_output, grad_mean, grad_invvar):\n        input_or_output, weight_, bias_, mean, invvar = ctx.saved_tensors\n        grad_input = grad_weight = grad_bias = None\n        grad_input, grad_weight, grad_bias = fused_layer_norm_affine_bwd(\n            grad_output,\n            mean,\n            invvar,\n            input_or_output,\n            ctx.normalized_shape,\n            weight_,\n            bias_,\n            ctx.eps,\n            ctx.memory_efficient,\n        )\n        return grad_input, grad_weight, grad_bias, None, None, None\n\n    def _fused_layer_norm_affine_setup_context(ctx, inputs, output):\n        input, weight, bias, normalized_shape, eps, memory_efficient = inputs\n        output, mean, invvar = output\n        input_ = input.contiguous()\n        weight_ = weight.contiguous()\n        bias_ = bias.contiguous()\n        if memory_efficient:\n            ctx.save_for_backward(output, weight_, bias_, None, invvar)\n        else:\n            ctx.save_for_backward(input_, weight_, bias_, mean, invvar)\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n\n    fused_layer_norm_affine_fwd.register_autograd(\n        _fused_layer_norm_affine_backward,\n        setup_context=_fused_layer_norm_affine_setup_context,\n    )\n\n\nclass FusedRMSNormAffineFunction(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, input, weight, normalized_shape, eps, memory_efficient=False):\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n        input_ = input.contiguous()\n        weight_ = weight.contiguous()\n        output, invvar = fused_layer_norm_cuda.rms_forward_affine(\n            input_, ctx.normalized_shape, weight_, ctx.eps\n        )\n        if ctx.memory_efficient:\n            ctx.save_for_backward(output, weight_, invvar)\n        else:\n            ctx.save_for_backward(input_, weight_, invvar)\n        return output\n\n    @staticmethod\n    def backward(ctx, grad_output):\n        input_or_output, weight_, invvar = ctx.saved_tensors\n        grad_input = grad_weight = None\n        grad_input, grad_weight = fused_layer_norm_cuda.rms_backward_affine(\n            grad_output.contiguous(),\n            invvar,\n            input_or_output,\n            ctx.normalized_shape,\n            weight_,\n            ctx.eps,\n            ctx.memory_efficient,\n        )\n        return grad_input, grad_weight, None, None, None\n\n\nif supports_custom_op():\n\n    @torch.library.custom_op(\"apex::fused_rms_norm_affine_fwd\", mutates_args=())\n    def fused_rms_norm_affine_fwd(\n        input: torch.Tensor,\n        weight: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n\n        input_ = input.contiguous()\n        weight_ = weight.contiguous()\n        output, invvar = fused_layer_norm_cuda.rms_forward_affine(\n            input_, normalized_shape, weight_, eps\n        )\n        return output, invvar\n\n    @fused_rms_norm_affine_fwd.register_fake\n    def fused_rms_norm_affine_fwd_fake(\n        input: torch.Tensor,\n        weight: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        input = input.contiguous()\n        weight = weight.contiguous()\n        idiff = input.ndim - len(normalized_shape)\n        n = 1\n        for i in range(idiff):\n            n *= input.shape[i]\n        if input.dtype in [torch.float16, torch.bfloat16]:\n            dtype = torch.float32\n        else:\n            dtype = input.dtype\n        return (\n            torch.empty_like(input),\n            torch.empty(\n                [n],\n                dtype=dtype,\n                device=input.device,\n                requires_grad=input.requires_grad,\n                memory_format=torch.contiguous_format,\n            ),\n        )\n\n    @torch.library.custom_op(\"apex::fused_rms_norm_affine_bwd\", mutates_args=())\n    def fused_rms_norm_affine_bwd(\n        grad_output: torch.Tensor,\n        invvar: torch.Tensor,\n        input_or_output: torch.Tensor,\n        normalized_shape: List[int],\n        weight: torch.Tensor,\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        grad_input, grad_weight = fused_layer_norm_cuda.rms_backward_affine(\n            grad_output.contiguous(),\n            invvar,\n            input_or_output,\n            normalized_shape,\n            weight,\n            eps,\n            memory_efficient,\n        )\n        return grad_input, grad_weight\n\n    @fused_rms_norm_affine_bwd.register_fake\n    def fused_rms_norm_affine_bwd_fake(\n        grad_output: torch.Tensor,\n        invvar: torch.Tensor,\n        input_or_output: torch.Tensor,\n        normalized_shape: List[int],\n        weight: torch.Tensor,\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        grad_input = torch.empty_like(input_or_output)\n        grad_weight = torch.empty_like(weight)\n        return grad_input, grad_weight\n\n    def _fused_rms_norm_affine_backward(ctx, grad_output, grad_invvar):\n        input_or_output, weight_, invvar = ctx.saved_tensors\n        grad_input = grad_weight = None\n        grad_input, grad_weight = fused_rms_norm_affine_bwd(\n            grad_output,\n            invvar,\n            input_or_output,\n            ctx.normalized_shape,\n            weight_,\n            ctx.eps,\n            ctx.memory_efficient,\n        )\n        return grad_input, grad_weight, None, None, None\n\n    def _fused_rms_norm_affine_setup_context(ctx, inputs, output):\n        input_, weight_, normalized_shape, eps, memory_efficient = inputs\n        output_, invvar = output\n        input_ = input_.contiguous()\n        weight_ = weight_.contiguous()\n        if memory_efficient:\n            ctx.save_for_backward(output_, weight_, invvar)\n        else:\n            ctx.save_for_backward(input_, weight_, invvar)\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n\n    fused_rms_norm_affine_fwd.register_autograd(\n        _fused_rms_norm_affine_backward,\n        setup_context=_fused_rms_norm_affine_setup_context,\n    )\n\n\nclass FusedLayerNormAffineMixedDtypesFunction(FusedLayerNormAffineFunction):\n    @staticmethod\n    def forward(ctx, input, weight, bias, normalized_shape, eps, memory_efficient=False):\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n        input_ = input.contiguous()\n        weight_ = weight.contiguous()\n        bias_ = bias.contiguous()\n        output, mean, invvar = fused_layer_norm_cuda.forward_affine_mixed_dtypes(\n            input_, ctx.normalized_shape, weight_, bias_, ctx.eps\n        )\n        if ctx.memory_efficient:\n            ctx.save_for_backward(output, weight_, bias_, None, invvar)\n        else:\n            ctx.save_for_backward(input_, weight_, bias_, mean, invvar)\n        return output\n\n\nclass FusedRMSNormAffineMixedDtypesFunction(FusedRMSNormAffineFunction):\n    @staticmethod\n    def forward(ctx, input, weight, normalized_shape, eps, memory_efficient=False):\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n        input_ = input.contiguous()\n        weight_ = weight.contiguous()\n        output, invvar = fused_layer_norm_cuda.rms_forward_affine_mixed_dtypes(\n            input_, ctx.normalized_shape, weight_, ctx.eps\n        )\n        if ctx.memory_efficient:\n            ctx.save_for_backward(output, weight_, invvar)\n        else:\n            ctx.save_for_backward(input_, weight_, invvar)\n        return output\n\n\nclass FusedLayerNormFunction(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, input, normalized_shape, eps, memory_efficient=False):\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n        input_ = input.contiguous()\n        output, mean, invvar = fused_layer_norm_cuda.forward(input_, ctx.normalized_shape, ctx.eps)\n        if ctx.memory_efficient:\n            ctx.save_for_backward(output, None, invvar)\n        else:\n            ctx.save_for_backward(input_, mean, invvar)\n        return output\n\n    @staticmethod\n    def backward(ctx, grad_output):\n        input_or_output, mean, invvar = ctx.saved_tensors\n        grad_input = fused_layer_norm_cuda.backward(\n            grad_output.contiguous(),\n            mean,\n            invvar,\n            input_or_output,\n            ctx.normalized_shape,\n            ctx.eps,\n            ctx.memory_efficient,\n        )\n        return grad_input, None, None, None\n\n\nif supports_custom_op():\n\n    @torch.library.custom_op(\"apex::fused_layer_norm_fwd\", mutates_args=())\n    def fused_layer_norm_fwd(\n        input: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n\n        input_ = input.contiguous()\n        output, mean, invvar = fused_layer_norm_cuda.forward(input_, normalized_shape, eps)\n        return output, mean, invvar\n\n    @fused_layer_norm_fwd.register_fake\n    def fused_layer_norm_fwd_fake(\n        input: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        input = input.contiguous()\n        idiff = input.ndim - len(normalized_shape)\n        n = 1\n        for i in range(idiff):\n            n *= input.shape[i]\n        if input.dtype in [torch.float16, torch.bfloat16]:\n            dtype = torch.float32\n        else:\n            dtype = input.dtype\n        mean = torch.empty([n], dtype=dtype, device=input.device)\n        invvar = torch.empty_like(mean)\n        return torch.empty_like(input), mean, invvar\n\n    @torch.library.custom_op(\"apex::fused_layer_norm_bwd\", mutates_args=())\n    def fused_layer_norm_bwd(\n        grad_output: torch.Tensor,\n        mean: torch.Tensor,\n        invvar: torch.Tensor,\n        input_or_output: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> torch.Tensor:\n        grad_input = fused_layer_norm_cuda.backward(\n            grad_output.contiguous(),\n            mean,\n            invvar,\n            input_or_output,\n            normalized_shape,\n            eps,\n            memory_efficient,\n        )\n        return grad_input\n\n    @fused_layer_norm_bwd.register_fake\n    def fused_layer_norm_bwd_fake(\n        grad_output: torch.Tensor,\n        mean: torch.Tensor,\n        invvar: torch.Tensor,\n        input_or_output: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> torch.Tensor:\n        grad_input = torch.empty_like(input_or_output)\n        return grad_input\n\n    def _fused_layer_norm_backward(ctx, grad_output, grad_mean, grad_invvar):\n        input_or_output, mean, invvar = ctx.saved_tensors\n        grad_input = fused_layer_norm_bwd(\n            grad_output,\n            mean,\n            invvar,\n            input_or_output,\n            ctx.normalized_shape,\n            ctx.eps,\n            ctx.memory_efficient,\n        )\n        return grad_input, None, None, None\n\n    def _fused_layer_norm_setup_context(ctx, inputs, output):\n        input, normalized_shape, eps, memory_efficient = inputs\n        output, mean, invvar = output\n        input_ = input.contiguous()\n        if memory_efficient:\n            ctx.save_for_backward(output, None, invvar)\n        else:\n            ctx.save_for_backward(input_, mean, invvar)\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n\n    fused_layer_norm_fwd.register_autograd(\n        _fused_layer_norm_backward,\n        setup_context=_fused_layer_norm_setup_context,\n    )\n\n\nclass FusedRMSNormFunction(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, input, normalized_shape, eps, memory_efficient=False):\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n        input_ = input.contiguous()\n        output, invvar = fused_layer_norm_cuda.rms_forward(input_, ctx.normalized_shape, ctx.eps)\n        if ctx.memory_efficient:\n            ctx.save_for_backward(output, invvar)\n        else:\n            ctx.save_for_backward(input_, invvar)\n        return output\n\n    @staticmethod\n    def backward(ctx, grad_output):\n        input_or_output, invvar = ctx.saved_tensors\n        grad_input = None\n        grad_input = fused_layer_norm_cuda.rms_backward(\n            grad_output.contiguous(),\n            invvar,\n            input_or_output,\n            ctx.normalized_shape,\n            ctx.eps,\n            ctx.memory_efficient,\n        )\n        return grad_input, None, None, None\n\n\nif supports_custom_op():\n\n    @torch.library.custom_op(\"apex::fused_rms_norm_fwd\", mutates_args=())\n    def fused_rms_norm_fwd(\n        input: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n\n        input_ = input.contiguous()\n        output, invvar = fused_layer_norm_cuda.rms_forward(input_, normalized_shape, eps)\n        return output, invvar\n\n    @fused_rms_norm_fwd.register_fake\n    def fused_rms_norm_fwd_fake(\n        input: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        input = input.contiguous()\n        idiff = input.ndim - len(normalized_shape)\n        n = 1\n        for i in range(idiff):\n            n *= input.shape[i]\n        if input.dtype in [torch.float16, torch.bfloat16]:\n            dtype = torch.float32\n        else:\n            dtype = input.dtype\n        return (\n            torch.empty_like(input),\n            torch.empty(\n                [n],\n                dtype=dtype,\n                device=input.device,\n                requires_grad=input.requires_grad,\n                memory_format=torch.contiguous_format,\n            ),\n        )\n\n    @torch.library.custom_op(\"apex::fused_rms_norm_bwd\", mutates_args=())\n    def fused_rms_norm_bwd(\n        grad_output: torch.Tensor,\n        invvar: torch.Tensor,\n        input_or_output: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> torch.Tensor:\n        grad_input = fused_layer_norm_cuda.rms_backward(\n            grad_output.contiguous(),\n            invvar,\n            input_or_output,\n            normalized_shape,\n            eps,\n            memory_efficient,\n        )\n        return grad_input\n\n    @fused_rms_norm_bwd.register_fake\n    def fused_rms_norm_bwd_fake(\n        grad_output: torch.Tensor,\n        invvar: torch.Tensor,\n        input_or_output: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> torch.Tensor:\n        grad_input = torch.empty_like(input_or_output)\n        return grad_input\n\n    def _fused_rms_norm_backward(ctx, grad_output, grad_invvar):\n        input_or_output, invvar = ctx.saved_tensors\n        grad_input = None\n        grad_input = fused_rms_norm_bwd(\n            grad_output,\n            invvar,\n            input_or_output,\n            ctx.normalized_shape,\n            ctx.eps,\n            ctx.memory_efficient,\n        )\n        return grad_input, None, None, None\n\n    def _fused_rms_norm_setup_context(ctx, inputs, output):\n        input_, normalized_shape, eps, memory_efficient = inputs\n        output_, invvar = output\n        input_ = input_.contiguous()\n        if memory_efficient:\n            ctx.save_for_backward(output_, invvar)\n        else:\n            ctx.save_for_backward(input_, invvar)\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n\n    fused_rms_norm_fwd.register_autograd(\n        _fused_rms_norm_backward, setup_context=_fused_rms_norm_setup_context\n    )\n\n\ndef fused_layer_norm_affine(\n    input, weight, bias, normalized_shape, eps=1e-6, memory_efficient=False\n):\n    args = _cast_if_autocast_enabled(input, weight, bias, normalized_shape, eps, memory_efficient)\n    with torch.amp.autocast(\"cuda\", enabled=False):\n        if supports_custom_op():\n            return fused_layer_norm_affine_fwd(*args)[0]\n        else:\n            return FusedLayerNormAffineFunction.apply(*args)\n\n\ndef fused_layer_norm(input, normalized_shape, eps=1e-6, memory_efficient=False):\n    args = _cast_if_autocast_enabled(input, normalized_shape, eps, memory_efficient)\n    with torch.amp.autocast(\"cuda\", enabled=False):\n        if supports_custom_op():\n            return fused_layer_norm_fwd(*args)[0]\n        else:\n            return FusedLayerNormFunction.apply(*args)\n\n\ndef mixed_dtype_fused_layer_norm_affine(\n    input, weight, bias, normalized_shape, eps=1e-6, memory_efficient=False\n):\n    args = _cast_if_autocast_enabled(input, weight, bias, normalized_shape, eps, memory_efficient)\n    with torch.amp.autocast(\"cuda\", enabled=False):\n        return FusedLayerNormAffineMixedDtypesFunction.apply(*args)\n\n\ndef fused_rms_norm_affine(input, weight, normalized_shape, eps=1e-6, memory_efficient=False):\n    args = _cast_if_autocast_enabled(input, weight, normalized_shape, eps, memory_efficient)\n    with torch.amp.autocast(\"cuda\", enabled=False):\n        if supports_custom_op():\n            return fused_rms_norm_affine_fwd(*args)[0]\n        else:\n            return FusedRMSNormAffineFunction.apply(*args)\n\n\ndef fused_rms_norm(input, normalized_shape, eps=1e-6, memory_efficient=False):\n    args = _cast_if_autocast_enabled(input, normalized_shape, eps, memory_efficient)\n    with torch.amp.autocast(\"cuda\", enabled=False):\n        if supports_custom_op():\n            return fused_rms_norm_fwd(*args)[0]\n        else:\n            return FusedRMSNormFunction.apply(*args)\n\n\ndef mixed_dtype_fused_rms_norm_affine(\n    input, weight, normalized_shape, eps=1e-6, memory_efficient=False\n):\n    args = _cast_if_autocast_enabled(input, weight, normalized_shape, eps, memory_efficient)\n    with torch.amp.autocast(\"cuda\", enabled=False):\n        return FusedRMSNormAffineMixedDtypesFunction.apply(*args)\n\n\nclass FusedLayerNorm(torch.nn.Module):\n    r\"\"\"Applies Layer Normalization over a mini-batch of inputs as described in\n    the paper `Layer Normalization`_ .\n\n    Currently only runs on cuda() tensors.\n\n    .. math::\n        y = \\frac{x - \\mathrm{E}[x]}{ \\sqrt{\\mathrm{Var}[x] + \\epsilon}} * \\gamma + \\beta\n\n    The mean and standard-deviation are calculated separately over the last\n    certain number dimensions which have to be of the shape specified by\n    :attr:`normalized_shape`.\n    :math:`\\gamma` and :math:`\\beta` are learnable affine transform parameters of\n    :attr:`normalized_shape` if :attr:`elementwise_affine` is ``True``.\n\n    .. note::\n        Unlike Batch Normalization and Instance Normalization, which applies\n        scalar scale and bias for each entire channel/plane with the\n        :attr:`affine` option, Layer Normalization applies per-element scale and\n        bias with :attr:`elementwise_affine`.\n\n    This layer uses statistics computed from input data in both training and\n    evaluation modes.\n\n    Args:\n        normalized_shape (int or list or torch.Size): input shape from an expected input\n            of size\n\n            .. math::\n                [* \\times \\text{normalized}\\_\\text{shape}[0] \\times \\text{normalized}\\_\\text{shape}[1]\n                    \\times \\ldots \\times \\text{normalized}\\_\\text{shape}[-1]]\n\n            If a single integer is used, it is treated as a singleton list, and this module will\n            normalize over the last dimension which is expected to be of that specific size.\n        eps: a value added to the denominator for numerical stability. Default: 1e-5\n        elementwise_affine: a boolean value that when set to ``True``, this module\n            has learnable per-element affine parameters initialized to ones (for weights)\n            and zeros (for biases). Default: ``True``.\n\n    Shape:\n        - Input: :math:`(N, *)`\n        - Output: :math:`(N, *)` (same shape as input)\n\n    Examples::\n\n        >>> input = torch.randn(20, 5, 10, 10)\n        >>> # With Learnable Parameters\n        >>> m = apex.normalization.FusedLayerNorm(input.size()[1:])\n        >>> # Without Learnable Parameters\n        >>> m = apex.normalization.FusedLayerNorm(input.size()[1:], elementwise_affine=False)\n        >>> # Normalize over last two dimensions\n        >>> m = apex.normalization.FusedLayerNorm([10, 10])\n        >>> # Normalize over last dimension of size 10\n        >>> m = apex.normalization.FusedLayerNorm(10)\n        >>> # Activating the module\n        >>> output = m(input)\n\n    .. _`Layer Normalization`: https://arxiv.org/abs/1607.06450\n    \"\"\"\n\n    def __init__(\n        self,\n        normalized_shape,\n        eps=1e-5,\n        elementwise_affine=True,\n        memory_efficient=False,\n    ):\n        super().__init__()\n\n        global fused_layer_norm_cuda\n        fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n\n        if isinstance(normalized_shape, numbers.Integral):\n            normalized_shape = (normalized_shape,)\n        self.normalized_shape = torch.Size(normalized_shape)\n        self.eps = eps\n        self.elementwise_affine = elementwise_affine\n        self.memory_efficient = memory_efficient\n        if self.elementwise_affine:\n            self.weight = Parameter(torch.empty(*normalized_shape))\n            self.bias = Parameter(torch.empty(*normalized_shape))\n        else:\n            self.register_parameter(\"weight\", None)\n            self.register_parameter(\"bias\", None)\n        self.reset_parameters()\n\n    def reset_parameters(self):\n        if self.elementwise_affine:\n            init.ones_(self.weight)\n            init.zeros_(self.bias)\n\n    def forward(self, input):\n        if (\n            torch.jit.is_tracing()\n            or torch.jit.is_scripting()\n            or torch.compiler.is_compiling()\n            or not input.is_cuda\n        ):\n            return F.layer_norm(input, self.normalized_shape, self.weight, self.bias, self.eps)\n        if self.elementwise_affine:\n            return fused_layer_norm_affine(\n                input,\n                self.weight,\n                self.bias,\n                self.normalized_shape,\n                self.eps,\n                self.memory_efficient,\n            )\n        else:\n            return fused_layer_norm(input, self.normalized_shape, self.eps, self.memory_efficient)\n\n    def extra_repr(self):\n        return \"{normalized_shape}, eps={eps}, elementwise_affine={elementwise_affine}\".format(\n            **self.__dict__\n        )\n\n\nclass FusedRMSNorm(torch.nn.Module):\n    r\"\"\"Applies RMS Normalization over a mini-batch of inputs\n\n    Currently only runs on cuda() tensors.\n\n    .. math::\n        y = \\frac{x}{\\mathrm{RMS}[x]} * \\gamma\n\n    The root-mean-square is calculated separately over the last\n    certain number dimensions which have to be of the shape specified by\n    :attr:`normalized_shape`.\n    :math:`\\gamma` is a learnable affine transform parameter of\n    :attr:`normalized_shape` if :attr:`elementwise_affine` is ``True``.\n    `epsilon` is added to the mean-square, then the root of the sum is taken.\n\n    .. note::\n        Unlike Batch Normalization and Instance Normalization, which applies\n        scalar scale and bias for each entire channel/plane with the\n        :attr:`affine` option, RMS Normalization applies per-element scale\n        with :attr:`elementwise_affine`.\n\n    This layer uses statistics computed from input data in both training and\n    evaluation modes.\n\n    Args:\n        normalized_shape (int or list or torch.Size): input shape from an expected input\n            of size\n\n            .. math::\n                [* \\times \\text{normalized}\\_\\text{shape}[0] \\times \\text{normalized}\\_\\text{shape}[1]\n                    \\times \\ldots \\times \\text{normalized}\\_\\text{shape}[-1]]\n\n            If a single integer is used, it is treated as a singleton list, and this module will\n            normalize over the last dimension which is expected to be of that specific size.\n        eps: a value added to the denominator for numerical stability. Default: 1e-5\n        elementwise_affine: a boolean value that when set to ``True``, this module\n            has learnable per-element affine parameters initialized to ones (for weights)\n            and zeros (for biases). Default: ``True``.\n\n    Shape:\n        - Input: :math:`(N, *)`\n        - Output: :math:`(N, *)` (same shape as input)\n\n    Examples::\n\n        >>> input = torch.randn(20, 5, 10, 10)\n        >>> # With Learnable Parameters\n        >>> m = apex.normalization.FusedRMSNorm(input.size()[1:])\n        >>> # Without Learnable Parameters\n        >>> m = apex.normalization.FusedRMSNorm(input.size()[1:], elementwise_affine=False)\n        >>> # Normalize over last two dimensions\n        >>> m = apex.normalization.FusedRMSNorm([10, 10])\n        >>> # Normalize over last dimension of size 10\n        >>> m = apex.normalization.FusedRMSNorm(10)\n        >>> # Activating the module\n        >>> output = m(input)\n\n    .. _`Root Mean Square Layer Normalization`: https://arxiv.org/pdf/1910.07467.pdf\n    \"\"\"\n\n    def __init__(\n        self,\n        normalized_shape,\n        eps=1e-5,\n        elementwise_affine=True,\n        memory_efficient=False,\n    ):\n        super().__init__()\n\n        global fused_layer_norm_cuda\n        fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n\n        if isinstance(normalized_shape, numbers.Integral):\n            normalized_shape = (normalized_shape,)\n        self.normalized_shape = torch.Size(normalized_shape)\n        self.eps = eps\n        self.elementwise_affine = elementwise_affine\n        self.memory_efficient = memory_efficient\n        if self.elementwise_affine:\n            self.weight = Parameter(torch.empty(*normalized_shape))\n        else:\n            self.register_parameter(\"weight\", None)\n        self.reset_parameters()\n\n    def reset_parameters(self):\n        if self.elementwise_affine:\n            init.ones_(self.weight)\n\n    def forward(self, input):\n        if (\n            torch.jit.is_tracing()\n            or torch.jit.is_scripting()\n            or torch.compiler.is_compiling()\n            or not input.is_cuda\n        ):\n            return manual_rms_norm(input, self.normalized_shape, self.weight, self.eps)\n\n        if self.elementwise_affine:\n            return fused_rms_norm_affine(\n                input,\n                self.weight,\n                self.normalized_shape,\n                self.eps,\n                self.memory_efficient,\n            )\n        else:\n            return fused_rms_norm(input, self.normalized_shape, self.eps, self.memory_efficient)\n\n    def extra_repr(self):\n        return \"{normalized_shape}, eps={eps}, elementwise_affine={elementwise_affine}\".format(\n            **self.__dict__\n        )\n\n\n# NOTE (mkozuki): Why \"mixed\"?\n# MixedFusedLayerNorm differs from FusedLayerNorm in that this layer norm uses parameter's dtype\n# as output tensor's dtype while FusedLayerNorm uses input tensor's dtype for output tensor's dtype.\n# See: `layer_norm_affine` and `layer_norm_affine_mixed_dtypes` in \"csrc/layer_norm_cuda.cpp\"\nclass MixedFusedLayerNorm(FusedLayerNorm):\n    def __init__(self, normalized_shape, eps=1e-5, *, memory_efficient=False, **kwargs):\n        if \"elementwise_affine\" in kwargs:\n            import warnings\n\n            warnings.warn(\"MixedFusedLayerNorm does not support `elementwise_affine` argument\")\n            elementwise_affine = kwargs.pop(\"elementwise_affine\")\n            if not elementwise_affine:\n                raise RuntimeError(\n                    \"MixedFusedLayerNorm does not support `elementwise_affine = False`\"\n                )\n\n        super().__init__(\n            normalized_shape=normalized_shape,\n            eps=eps,\n            elementwise_affine=True,\n            memory_efficient=memory_efficient,\n        )\n\n    def forward(self, input: torch.Tensor):\n        # NOTE (mkozuki): CPU path is here mainly for unittest sake.\n        if (\n            torch.jit.is_tracing()\n            or torch.jit.is_scripting()\n            or torch.compiler.is_compiling()\n            or not input.is_cuda\n        ):\n            return F.layer_norm(input, self.normalized_shape, self.weight, self.bias, self.eps)\n        return mixed_dtype_fused_layer_norm_affine(\n            input,\n            self.weight,\n            self.bias,\n            self.normalized_shape,\n            self.eps,\n            self.memory_efficient,\n        )\n\n\n# MixedFusedLayerNorm differs from FusedLayerNorm in that this layer norm uses parameter's dtype\n# as output tensor's dtype while FusedLayerNorm uses input tensor's dtype for output tensor's dtype.\n# See: `layer_norm_affine` and `layer_norm_affine_mixed_dtypes` in \"csrc/layer_norm_cuda.cpp\"\nclass MixedFusedRMSNorm(FusedRMSNorm):\n    def __init__(self, normalized_shape, eps=1e-5, *, memory_efficient=False, **kwargs):\n        if \"elementwise_affine\" in kwargs:\n            import warnings\n\n            warnings.warn(\"MixedFusedRMSNorm does not support `elementwise_affine` argument\")\n            elementwise_affine = kwargs.pop(\"elementwise_affine\")\n            if not elementwise_affine:\n                raise RuntimeError(\n                    \"MixedFusedRMSNorm does not support `elementwise_affine = False`\"\n                )\n\n        super().__init__(\n            normalized_shape=normalized_shape,\n            eps=eps,\n            elementwise_affine=True,\n            memory_efficient=memory_efficient,\n        )\n\n    def forward(self, input: torch.Tensor):\n        # NOTE (mkozuki): CPU path is here mainly for unittest sake.\n        # TODO Manual RMS Norm Implementation Here\n        if (\n            torch.jit.is_tracing()\n            or torch.jit.is_scripting()\n            or torch.compiler.is_compiling()\n            or not input.is_cuda\n        ):\n            return manual_rms_norm(input, self.normalized_shape, self.weight, self.eps)\n        return mixed_dtype_fused_rms_norm_affine(\n            input, self.weight, self.normalized_shape, self.eps, self.memory_efficient\n        )\n")


In [ ]:
pathlib.Path("head.py").write_text("import importlib\nimport numbers\n\nimport torch\nfrom torch.nn.parameter import Parameter\nfrom torch.nn import init\nfrom torch.nn import functional as F\nfrom typing import List, Tuple\n\nfrom apex._autocast_utils import _cast_if_autocast_enabled\n\nglobal fused_layer_norm_cuda\nfused_layer_norm_cuda = None\n\n\n# PyTorch supports `torch.library.custom_op` since 2.4.0.\ndef supports_custom_op() -> bool:\n    return hasattr(torch.library, \"custom_op\")\n\n\n# Reference implementation from Huggingface\ndef manual_rms_norm(input, normalized_shape, weight, eps):\n    # layer norm should always be calculated in float32\n    dims = tuple(i for i in range(-1, -len(normalized_shape) - 1, -1))\n    variance = input.to(torch.float32).pow(2).mean(dims, keepdim=True)\n    input = input * torch.rsqrt(variance + eps)\n\n    if weight is None:\n        return input\n\n    # convert into half-precision if necessary\n    if weight.dtype in [torch.float16, torch.bfloat16]:\n        input = input.to(weight.dtype)\n\n    return weight * input\n\n\nclass FusedLayerNormAffineFunction(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, input, weight, bias, normalized_shape, eps, memory_efficient=False):\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n        input_ = input.contiguous()\n        weight_ = weight.contiguous()\n        bias_ = bias.contiguous()\n        output, mean, invvar = fused_layer_norm_cuda.forward_affine(\n            input_, ctx.normalized_shape, weight_, bias_, ctx.eps\n        )\n        if ctx.memory_efficient:\n            ctx.save_for_backward(output, weight_, bias_, None, invvar)\n        else:\n            ctx.save_for_backward(input_, weight_, bias_, mean, invvar)\n        return output\n\n    @staticmethod\n    def backward(ctx, grad_output):\n        input_or_output, weight_, bias_, mean, invvar = ctx.saved_tensors\n        grad_input = grad_weight = grad_bias = None\n        grad_input, grad_weight, grad_bias = fused_layer_norm_cuda.backward_affine(\n            grad_output.contiguous(),\n            mean,\n            invvar,\n            input_or_output,\n            ctx.normalized_shape,\n            weight_,\n            bias_,\n            ctx.eps,\n            ctx.memory_efficient,\n        )\n        return grad_input, grad_weight, grad_bias, None, None, None\n\n\nif supports_custom_op():\n\n    @torch.library.custom_op(\"apex::fused_layer_norm_affine_fwd\", mutates_args=())\n    def fused_layer_norm_affine_fwd(\n        input: torch.Tensor,\n        weight: torch.Tensor,\n        bias: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n\n        input_ = input.contiguous()\n        weight_ = weight.contiguous()\n        bias_ = bias.contiguous()\n        output, mean, invvar = fused_layer_norm_cuda.forward_affine(\n            input_, normalized_shape, weight_, bias_, eps\n        )\n        return output, mean, invvar\n\n    @fused_layer_norm_affine_fwd.register_fake\n    def fused_layer_norm_affine_fwd_fake(\n        input: torch.Tensor,\n        weight: torch.Tensor,\n        bias: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        input = input.contiguous()\n        weight = weight.contiguous()\n        bias = bias.contiguous()\n        idiff = input.ndim - len(normalized_shape)\n        n = 1\n        for i in range(idiff):\n            n *= input.shape[i]\n        if input.dtype in [torch.float16, torch.bfloat16]:\n            dtype = torch.float32\n        else:\n            dtype = input.dtype\n        mean = torch.empty([n], dtype=dtype, device=input.device)\n        invvar = torch.empty_like(mean)\n        return torch.empty_like(input), mean, invvar\n\n    @torch.library.custom_op(\"apex::fused_layer_norm_affine_bwd\", mutates_args=())\n    def fused_layer_norm_affine_bwd(\n        grad_output: torch.Tensor,\n        mean: torch.Tensor,\n        invvar: torch.Tensor,\n        input_or_output: torch.Tensor,\n        normalized_shape: List[int],\n        weight: torch.Tensor,\n        bias: torch.Tensor,\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        grad_input, grad_weight, grad_bias = fused_layer_norm_cuda.backward_affine(\n            grad_output.contiguous(),\n            mean,\n            invvar,\n            input_or_output,\n            normalized_shape,\n            weight,\n            bias,\n            eps,\n            memory_efficient,\n        )\n        return grad_input, grad_weight, grad_bias\n\n    @fused_layer_norm_affine_bwd.register_fake\n    def fused_layer_norm_affine_bwd_fake(\n        grad_output: torch.Tensor,\n        mean: torch.Tensor,\n        invvar: torch.Tensor,\n        input_or_output: torch.Tensor,\n        normalized_shape: List[int],\n        weight: torch.Tensor,\n        bias: torch.Tensor,\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        grad_input = torch.empty_like(input_or_output)\n        grad_weight = torch.empty_like(weight)\n        grad_bias = torch.empty_like(bias)\n        return grad_input, grad_weight, grad_bias\n\n    def _fused_layer_norm_affine_backward(ctx, grad_output, grad_mean, grad_invvar):\n        input_or_output, weight_, bias_, mean, invvar = ctx.saved_tensors\n        grad_input = grad_weight = grad_bias = None\n        grad_input, grad_weight, grad_bias = fused_layer_norm_affine_bwd(\n            grad_output,\n            mean,\n            invvar,\n            input_or_output,\n            ctx.normalized_shape,\n            weight_,\n            bias_,\n            ctx.eps,\n            ctx.memory_efficient,\n        )\n        return grad_input, grad_weight, grad_bias, None, None, None\n\n    def _fused_layer_norm_affine_setup_context(ctx, inputs, output):\n        input, weight, bias, normalized_shape, eps, memory_efficient = inputs\n        output, mean, invvar = output\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n        # See _fused_rms_norm_affine_setup_context: skip the saves under no_grad to\n        # avoid leaking the retained tensors (issue #1999).\n        if not torch.is_grad_enabled():\n            return\n        input_ = input.contiguous()\n        weight_ = weight.contiguous()\n        bias_ = bias.contiguous()\n        if memory_efficient:\n            ctx.save_for_backward(output, weight_, bias_, None, invvar)\n        else:\n            ctx.save_for_backward(input_, weight_, bias_, mean, invvar)\n\n    fused_layer_norm_affine_fwd.register_autograd(\n        _fused_layer_norm_affine_backward,\n        setup_context=_fused_layer_norm_affine_setup_context,\n    )\n\n\nclass FusedRMSNormAffineFunction(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, input, weight, normalized_shape, eps, memory_efficient=False):\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n        input_ = input.contiguous()\n        weight_ = weight.contiguous()\n        output, invvar = fused_layer_norm_cuda.rms_forward_affine(\n            input_, ctx.normalized_shape, weight_, ctx.eps\n        )\n        if ctx.memory_efficient:\n            ctx.save_for_backward(output, weight_, invvar)\n        else:\n            ctx.save_for_backward(input_, weight_, invvar)\n        return output\n\n    @staticmethod\n    def backward(ctx, grad_output):\n        input_or_output, weight_, invvar = ctx.saved_tensors\n        grad_input = grad_weight = None\n        grad_input, grad_weight = fused_layer_norm_cuda.rms_backward_affine(\n            grad_output.contiguous(),\n            invvar,\n            input_or_output,\n            ctx.normalized_shape,\n            weight_,\n            ctx.eps,\n            ctx.memory_efficient,\n        )\n        return grad_input, grad_weight, None, None, None\n\n\nif supports_custom_op():\n\n    @torch.library.custom_op(\"apex::fused_rms_norm_affine_fwd\", mutates_args=())\n    def fused_rms_norm_affine_fwd(\n        input: torch.Tensor,\n        weight: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n\n        input_ = input.contiguous()\n        weight_ = weight.contiguous()\n        output, invvar = fused_layer_norm_cuda.rms_forward_affine(\n            input_, normalized_shape, weight_, eps\n        )\n        return output, invvar\n\n    @fused_rms_norm_affine_fwd.register_fake\n    def fused_rms_norm_affine_fwd_fake(\n        input: torch.Tensor,\n        weight: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        input = input.contiguous()\n        weight = weight.contiguous()\n        idiff = input.ndim - len(normalized_shape)\n        n = 1\n        for i in range(idiff):\n            n *= input.shape[i]\n        if input.dtype in [torch.float16, torch.bfloat16]:\n            dtype = torch.float32\n        else:\n            dtype = input.dtype\n        return (\n            torch.empty_like(input),\n            torch.empty(\n                [n],\n                dtype=dtype,\n                device=input.device,\n                requires_grad=input.requires_grad,\n                memory_format=torch.contiguous_format,\n            ),\n        )\n\n    @torch.library.custom_op(\"apex::fused_rms_norm_affine_bwd\", mutates_args=())\n    def fused_rms_norm_affine_bwd(\n        grad_output: torch.Tensor,\n        invvar: torch.Tensor,\n        input_or_output: torch.Tensor,\n        normalized_shape: List[int],\n        weight: torch.Tensor,\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        grad_input, grad_weight = fused_layer_norm_cuda.rms_backward_affine(\n            grad_output.contiguous(),\n            invvar,\n            input_or_output,\n            normalized_shape,\n            weight,\n            eps,\n            memory_efficient,\n        )\n        return grad_input, grad_weight\n\n    @fused_rms_norm_affine_bwd.register_fake\n    def fused_rms_norm_affine_bwd_fake(\n        grad_output: torch.Tensor,\n        invvar: torch.Tensor,\n        input_or_output: torch.Tensor,\n        normalized_shape: List[int],\n        weight: torch.Tensor,\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        grad_input = torch.empty_like(input_or_output)\n        grad_weight = torch.empty_like(weight)\n        return grad_input, grad_weight\n\n    def _fused_rms_norm_affine_backward(ctx, grad_output, grad_invvar):\n        input_or_output, weight_, invvar = ctx.saved_tensors\n        grad_input = grad_weight = None\n        grad_input, grad_weight = fused_rms_norm_affine_bwd(\n            grad_output,\n            invvar,\n            input_or_output,\n            ctx.normalized_shape,\n            weight_,\n            ctx.eps,\n            ctx.memory_efficient,\n        )\n        return grad_input, grad_weight, None, None, None\n\n    def _fused_rms_norm_affine_setup_context(ctx, inputs, output):\n        input_, weight_, normalized_shape, eps, memory_efficient = inputs\n        output_, invvar = output\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n        # Under torch.no_grad() backward will never run, so don't retain the\n        # activation/invvar tensors. For custom ops these saves are held in autograd\n        # metadata that isn't released after the call returns, which otherwise leaks\n        # two CUDA tensors per forward in inference loops (issue #1999).\n        if not torch.is_grad_enabled():\n            return\n        input_ = input_.contiguous()\n        weight_ = weight_.contiguous()\n        if memory_efficient:\n            ctx.save_for_backward(output_, weight_, invvar)\n        else:\n            ctx.save_for_backward(input_, weight_, invvar)\n\n    fused_rms_norm_affine_fwd.register_autograd(\n        _fused_rms_norm_affine_backward,\n        setup_context=_fused_rms_norm_affine_setup_context,\n    )\n\n\nclass FusedLayerNormAffineMixedDtypesFunction(FusedLayerNormAffineFunction):\n    @staticmethod\n    def forward(ctx, input, weight, bias, normalized_shape, eps, memory_efficient=False):\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n        input_ = input.contiguous()\n        weight_ = weight.contiguous()\n        bias_ = bias.contiguous()\n        output, mean, invvar = fused_layer_norm_cuda.forward_affine_mixed_dtypes(\n            input_, ctx.normalized_shape, weight_, bias_, ctx.eps\n        )\n        if ctx.memory_efficient:\n            ctx.save_for_backward(output, weight_, bias_, None, invvar)\n        else:\n            ctx.save_for_backward(input_, weight_, bias_, mean, invvar)\n        return output\n\n\nclass FusedRMSNormAffineMixedDtypesFunction(FusedRMSNormAffineFunction):\n    @staticmethod\n    def forward(ctx, input, weight, normalized_shape, eps, memory_efficient=False):\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n        input_ = input.contiguous()\n        weight_ = weight.contiguous()\n        output, invvar = fused_layer_norm_cuda.rms_forward_affine_mixed_dtypes(\n            input_, ctx.normalized_shape, weight_, ctx.eps\n        )\n        if ctx.memory_efficient:\n            ctx.save_for_backward(output, weight_, invvar)\n        else:\n            ctx.save_for_backward(input_, weight_, invvar)\n        return output\n\n\nclass FusedLayerNormFunction(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, input, normalized_shape, eps, memory_efficient=False):\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n        input_ = input.contiguous()\n        output, mean, invvar = fused_layer_norm_cuda.forward(input_, ctx.normalized_shape, ctx.eps)\n        if ctx.memory_efficient:\n            ctx.save_for_backward(output, None, invvar)\n        else:\n            ctx.save_for_backward(input_, mean, invvar)\n        return output\n\n    @staticmethod\n    def backward(ctx, grad_output):\n        input_or_output, mean, invvar = ctx.saved_tensors\n        grad_input = fused_layer_norm_cuda.backward(\n            grad_output.contiguous(),\n            mean,\n            invvar,\n            input_or_output,\n            ctx.normalized_shape,\n            ctx.eps,\n            ctx.memory_efficient,\n        )\n        return grad_input, None, None, None\n\n\nif supports_custom_op():\n\n    @torch.library.custom_op(\"apex::fused_layer_norm_fwd\", mutates_args=())\n    def fused_layer_norm_fwd(\n        input: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n\n        input_ = input.contiguous()\n        output, mean, invvar = fused_layer_norm_cuda.forward(input_, normalized_shape, eps)\n        return output, mean, invvar\n\n    @fused_layer_norm_fwd.register_fake\n    def fused_layer_norm_fwd_fake(\n        input: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        input = input.contiguous()\n        idiff = input.ndim - len(normalized_shape)\n        n = 1\n        for i in range(idiff):\n            n *= input.shape[i]\n        if input.dtype in [torch.float16, torch.bfloat16]:\n            dtype = torch.float32\n        else:\n            dtype = input.dtype\n        mean = torch.empty([n], dtype=dtype, device=input.device)\n        invvar = torch.empty_like(mean)\n        return torch.empty_like(input), mean, invvar\n\n    @torch.library.custom_op(\"apex::fused_layer_norm_bwd\", mutates_args=())\n    def fused_layer_norm_bwd(\n        grad_output: torch.Tensor,\n        mean: torch.Tensor,\n        invvar: torch.Tensor,\n        input_or_output: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> torch.Tensor:\n        grad_input = fused_layer_norm_cuda.backward(\n            grad_output.contiguous(),\n            mean,\n            invvar,\n            input_or_output,\n            normalized_shape,\n            eps,\n            memory_efficient,\n        )\n        return grad_input\n\n    @fused_layer_norm_bwd.register_fake\n    def fused_layer_norm_bwd_fake(\n        grad_output: torch.Tensor,\n        mean: torch.Tensor,\n        invvar: torch.Tensor,\n        input_or_output: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> torch.Tensor:\n        grad_input = torch.empty_like(input_or_output)\n        return grad_input\n\n    def _fused_layer_norm_backward(ctx, grad_output, grad_mean, grad_invvar):\n        input_or_output, mean, invvar = ctx.saved_tensors\n        grad_input = fused_layer_norm_bwd(\n            grad_output,\n            mean,\n            invvar,\n            input_or_output,\n            ctx.normalized_shape,\n            ctx.eps,\n            ctx.memory_efficient,\n        )\n        return grad_input, None, None, None\n\n    def _fused_layer_norm_setup_context(ctx, inputs, output):\n        input, normalized_shape, eps, memory_efficient = inputs\n        output, mean, invvar = output\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n        # See _fused_rms_norm_affine_setup_context: skip the saves under no_grad to\n        # avoid leaking the retained tensors (issue #1999).\n        if not torch.is_grad_enabled():\n            return\n        input_ = input.contiguous()\n        if memory_efficient:\n            ctx.save_for_backward(output, None, invvar)\n        else:\n            ctx.save_for_backward(input_, mean, invvar)\n\n    fused_layer_norm_fwd.register_autograd(\n        _fused_layer_norm_backward,\n        setup_context=_fused_layer_norm_setup_context,\n    )\n\n\nclass FusedRMSNormFunction(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, input, normalized_shape, eps, memory_efficient=False):\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n        input_ = input.contiguous()\n        output, invvar = fused_layer_norm_cuda.rms_forward(input_, ctx.normalized_shape, ctx.eps)\n        if ctx.memory_efficient:\n            ctx.save_for_backward(output, invvar)\n        else:\n            ctx.save_for_backward(input_, invvar)\n        return output\n\n    @staticmethod\n    def backward(ctx, grad_output):\n        input_or_output, invvar = ctx.saved_tensors\n        grad_input = None\n        grad_input = fused_layer_norm_cuda.rms_backward(\n            grad_output.contiguous(),\n            invvar,\n            input_or_output,\n            ctx.normalized_shape,\n            ctx.eps,\n            ctx.memory_efficient,\n        )\n        return grad_input, None, None, None\n\n\nif supports_custom_op():\n\n    @torch.library.custom_op(\"apex::fused_rms_norm_fwd\", mutates_args=())\n    def fused_rms_norm_fwd(\n        input: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        global fused_layer_norm_cuda\n        if fused_layer_norm_cuda is None:\n            fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n\n        input_ = input.contiguous()\n        output, invvar = fused_layer_norm_cuda.rms_forward(input_, normalized_shape, eps)\n        return output, invvar\n\n    @fused_rms_norm_fwd.register_fake\n    def fused_rms_norm_fwd_fake(\n        input: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        input = input.contiguous()\n        idiff = input.ndim - len(normalized_shape)\n        n = 1\n        for i in range(idiff):\n            n *= input.shape[i]\n        if input.dtype in [torch.float16, torch.bfloat16]:\n            dtype = torch.float32\n        else:\n            dtype = input.dtype\n        return (\n            torch.empty_like(input),\n            torch.empty(\n                [n],\n                dtype=dtype,\n                device=input.device,\n                requires_grad=input.requires_grad,\n                memory_format=torch.contiguous_format,\n            ),\n        )\n\n    @torch.library.custom_op(\"apex::fused_rms_norm_bwd\", mutates_args=())\n    def fused_rms_norm_bwd(\n        grad_output: torch.Tensor,\n        invvar: torch.Tensor,\n        input_or_output: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> torch.Tensor:\n        grad_input = fused_layer_norm_cuda.rms_backward(\n            grad_output.contiguous(),\n            invvar,\n            input_or_output,\n            normalized_shape,\n            eps,\n            memory_efficient,\n        )\n        return grad_input\n\n    @fused_rms_norm_bwd.register_fake\n    def fused_rms_norm_bwd_fake(\n        grad_output: torch.Tensor,\n        invvar: torch.Tensor,\n        input_or_output: torch.Tensor,\n        normalized_shape: List[int],\n        eps: float,\n        memory_efficient: bool = False,\n    ) -> torch.Tensor:\n        grad_input = torch.empty_like(input_or_output)\n        return grad_input\n\n    def _fused_rms_norm_backward(ctx, grad_output, grad_invvar):\n        input_or_output, invvar = ctx.saved_tensors\n        grad_input = None\n        grad_input = fused_rms_norm_bwd(\n            grad_output,\n            invvar,\n            input_or_output,\n            ctx.normalized_shape,\n            ctx.eps,\n            ctx.memory_efficient,\n        )\n        return grad_input, None, None, None\n\n    def _fused_rms_norm_setup_context(ctx, inputs, output):\n        input_, normalized_shape, eps, memory_efficient = inputs\n        output_, invvar = output\n        ctx.normalized_shape = normalized_shape\n        ctx.eps = eps\n        ctx.memory_efficient = memory_efficient\n        # See _fused_rms_norm_affine_setup_context: skip the saves under no_grad to\n        # avoid leaking the retained tensors (issue #1999).\n        if not torch.is_grad_enabled():\n            return\n        input_ = input_.contiguous()\n        if memory_efficient:\n            ctx.save_for_backward(output_, invvar)\n        else:\n            ctx.save_for_backward(input_, invvar)\n\n    fused_rms_norm_fwd.register_autograd(\n        _fused_rms_norm_backward, setup_context=_fused_rms_norm_setup_context\n    )\n\n\ndef fused_layer_norm_affine(\n    input, weight, bias, normalized_shape, eps=1e-6, memory_efficient=False\n):\n    args = _cast_if_autocast_enabled(input, weight, bias, normalized_shape, eps, memory_efficient)\n    with torch.amp.autocast(\"cuda\", enabled=False):\n        if supports_custom_op():\n            return fused_layer_norm_affine_fwd(*args)[0]\n        else:\n            return FusedLayerNormAffineFunction.apply(*args)\n\n\ndef fused_layer_norm(input, normalized_shape, eps=1e-6, memory_efficient=False):\n    args = _cast_if_autocast_enabled(input, normalized_shape, eps, memory_efficient)\n    with torch.amp.autocast(\"cuda\", enabled=False):\n        if supports_custom_op():\n            return fused_layer_norm_fwd(*args)[0]\n        else:\n            return FusedLayerNormFunction.apply(*args)\n\n\ndef mixed_dtype_fused_layer_norm_affine(\n    input, weight, bias, normalized_shape, eps=1e-6, memory_efficient=False\n):\n    args = _cast_if_autocast_enabled(input, weight, bias, normalized_shape, eps, memory_efficient)\n    with torch.amp.autocast(\"cuda\", enabled=False):\n        return FusedLayerNormAffineMixedDtypesFunction.apply(*args)\n\n\ndef fused_rms_norm_affine(input, weight, normalized_shape, eps=1e-6, memory_efficient=False):\n    args = _cast_if_autocast_enabled(input, weight, normalized_shape, eps, memory_efficient)\n    with torch.amp.autocast(\"cuda\", enabled=False):\n        if supports_custom_op():\n            return fused_rms_norm_affine_fwd(*args)[0]\n        else:\n            return FusedRMSNormAffineFunction.apply(*args)\n\n\ndef fused_rms_norm(input, normalized_shape, eps=1e-6, memory_efficient=False):\n    args = _cast_if_autocast_enabled(input, normalized_shape, eps, memory_efficient)\n    with torch.amp.autocast(\"cuda\", enabled=False):\n        if supports_custom_op():\n            return fused_rms_norm_fwd(*args)[0]\n        else:\n            return FusedRMSNormFunction.apply(*args)\n\n\ndef mixed_dtype_fused_rms_norm_affine(\n    input, weight, normalized_shape, eps=1e-6, memory_efficient=False\n):\n    args = _cast_if_autocast_enabled(input, weight, normalized_shape, eps, memory_efficient)\n    with torch.amp.autocast(\"cuda\", enabled=False):\n        return FusedRMSNormAffineMixedDtypesFunction.apply(*args)\n\n\nclass FusedLayerNorm(torch.nn.Module):\n    r\"\"\"Applies Layer Normalization over a mini-batch of inputs as described in\n    the paper `Layer Normalization`_ .\n\n    Currently only runs on cuda() tensors.\n\n    .. math::\n        y = \\frac{x - \\mathrm{E}[x]}{ \\sqrt{\\mathrm{Var}[x] + \\epsilon}} * \\gamma + \\beta\n\n    The mean and standard-deviation are calculated separately over the last\n    certain number dimensions which have to be of the shape specified by\n    :attr:`normalized_shape`.\n    :math:`\\gamma` and :math:`\\beta` are learnable affine transform parameters of\n    :attr:`normalized_shape` if :attr:`elementwise_affine` is ``True``.\n\n    .. note::\n        Unlike Batch Normalization and Instance Normalization, which applies\n        scalar scale and bias for each entire channel/plane with the\n        :attr:`affine` option, Layer Normalization applies per-element scale and\n        bias with :attr:`elementwise_affine`.\n\n    This layer uses statistics computed from input data in both training and\n    evaluation modes.\n\n    Args:\n        normalized_shape (int or list or torch.Size): input shape from an expected input\n            of size\n\n            .. math::\n                [* \\times \\text{normalized}\\_\\text{shape}[0] \\times \\text{normalized}\\_\\text{shape}[1]\n                    \\times \\ldots \\times \\text{normalized}\\_\\text{shape}[-1]]\n\n            If a single integer is used, it is treated as a singleton list, and this module will\n            normalize over the last dimension which is expected to be of that specific size.\n        eps: a value added to the denominator for numerical stability. Default: 1e-5\n        elementwise_affine: a boolean value that when set to ``True``, this module\n            has learnable per-element affine parameters initialized to ones (for weights)\n            and zeros (for biases). Default: ``True``.\n\n    Shape:\n        - Input: :math:`(N, *)`\n        - Output: :math:`(N, *)` (same shape as input)\n\n    Examples::\n\n        >>> input = torch.randn(20, 5, 10, 10)\n        >>> # With Learnable Parameters\n        >>> m = apex.normalization.FusedLayerNorm(input.size()[1:])\n        >>> # Without Learnable Parameters\n        >>> m = apex.normalization.FusedLayerNorm(input.size()[1:], elementwise_affine=False)\n        >>> # Normalize over last two dimensions\n        >>> m = apex.normalization.FusedLayerNorm([10, 10])\n        >>> # Normalize over last dimension of size 10\n        >>> m = apex.normalization.FusedLayerNorm(10)\n        >>> # Activating the module\n        >>> output = m(input)\n\n    .. _`Layer Normalization`: https://arxiv.org/abs/1607.06450\n    \"\"\"\n\n    def __init__(\n        self,\n        normalized_shape,\n        eps=1e-5,\n        elementwise_affine=True,\n        memory_efficient=False,\n    ):\n        super().__init__()\n\n        global fused_layer_norm_cuda\n        fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n\n        if isinstance(normalized_shape, numbers.Integral):\n            normalized_shape = (normalized_shape,)\n        self.normalized_shape = torch.Size(normalized_shape)\n        self.eps = eps\n        self.elementwise_affine = elementwise_affine\n        self.memory_efficient = memory_efficient\n        if self.elementwise_affine:\n            self.weight = Parameter(torch.empty(*normalized_shape))\n            self.bias = Parameter(torch.empty(*normalized_shape))\n        else:\n            self.register_parameter(\"weight\", None)\n            self.register_parameter(\"bias\", None)\n        self.reset_parameters()\n\n    def reset_parameters(self):\n        if self.elementwise_affine:\n            init.ones_(self.weight)\n            init.zeros_(self.bias)\n\n    def forward(self, input):\n        if (\n            torch.jit.is_tracing()\n            or torch.jit.is_scripting()\n            or torch.compiler.is_compiling()\n            or not input.is_cuda\n        ):\n            return F.layer_norm(input, self.normalized_shape, self.weight, self.bias, self.eps)\n        if self.elementwise_affine:\n            return fused_layer_norm_affine(\n                input,\n                self.weight,\n                self.bias,\n                self.normalized_shape,\n                self.eps,\n                self.memory_efficient,\n            )\n        else:\n            return fused_layer_norm(input, self.normalized_shape, self.eps, self.memory_efficient)\n\n    def extra_repr(self):\n        return \"{normalized_shape}, eps={eps}, elementwise_affine={elementwise_affine}\".format(\n            **self.__dict__\n        )\n\n\nclass FusedRMSNorm(torch.nn.Module):\n    r\"\"\"Applies RMS Normalization over a mini-batch of inputs\n\n    Currently only runs on cuda() tensors.\n\n    .. math::\n        y = \\frac{x}{\\mathrm{RMS}[x]} * \\gamma\n\n    The root-mean-square is calculated separately over the last\n    certain number dimensions which have to be of the shape specified by\n    :attr:`normalized_shape`.\n    :math:`\\gamma` is a learnable affine transform parameter of\n    :attr:`normalized_shape` if :attr:`elementwise_affine` is ``True``.\n    `epsilon` is added to the mean-square, then the root of the sum is taken.\n\n    .. note::\n        Unlike Batch Normalization and Instance Normalization, which applies\n        scalar scale and bias for each entire channel/plane with the\n        :attr:`affine` option, RMS Normalization applies per-element scale\n        with :attr:`elementwise_affine`.\n\n    This layer uses statistics computed from input data in both training and\n    evaluation modes.\n\n    Args:\n        normalized_shape (int or list or torch.Size): input shape from an expected input\n            of size\n\n            .. math::\n                [* \\times \\text{normalized}\\_\\text{shape}[0] \\times \\text{normalized}\\_\\text{shape}[1]\n                    \\times \\ldots \\times \\text{normalized}\\_\\text{shape}[-1]]\n\n            If a single integer is used, it is treated as a singleton list, and this module will\n            normalize over the last dimension which is expected to be of that specific size.\n        eps: a value added to the denominator for numerical stability. Default: 1e-5\n        elementwise_affine: a boolean value that when set to ``True``, this module\n            has learnable per-element affine parameters initialized to ones (for weights)\n            and zeros (for biases). Default: ``True``.\n\n    Shape:\n        - Input: :math:`(N, *)`\n        - Output: :math:`(N, *)` (same shape as input)\n\n    Examples::\n\n        >>> input = torch.randn(20, 5, 10, 10)\n        >>> # With Learnable Parameters\n        >>> m = apex.normalization.FusedRMSNorm(input.size()[1:])\n        >>> # Without Learnable Parameters\n        >>> m = apex.normalization.FusedRMSNorm(input.size()[1:], elementwise_affine=False)\n        >>> # Normalize over last two dimensions\n        >>> m = apex.normalization.FusedRMSNorm([10, 10])\n        >>> # Normalize over last dimension of size 10\n        >>> m = apex.normalization.FusedRMSNorm(10)\n        >>> # Activating the module\n        >>> output = m(input)\n\n    .. _`Root Mean Square Layer Normalization`: https://arxiv.org/pdf/1910.07467.pdf\n    \"\"\"\n\n    def __init__(\n        self,\n        normalized_shape,\n        eps=1e-5,\n        elementwise_affine=True,\n        memory_efficient=False,\n    ):\n        super().__init__()\n\n        global fused_layer_norm_cuda\n        fused_layer_norm_cuda = importlib.import_module(\"fused_layer_norm_cuda\")\n\n        if isinstance(normalized_shape, numbers.Integral):\n            normalized_shape = (normalized_shape,)\n        self.normalized_shape = torch.Size(normalized_shape)\n        self.eps = eps\n        self.elementwise_affine = elementwise_affine\n        self.memory_efficient = memory_efficient\n        if self.elementwise_affine:\n            self.weight = Parameter(torch.empty(*normalized_shape))\n        else:\n            self.register_parameter(\"weight\", None)\n        self.reset_parameters()\n\n    def reset_parameters(self):\n        if self.elementwise_affine:\n            init.ones_(self.weight)\n\n    def forward(self, input):\n        if (\n            torch.jit.is_tracing()\n            or torch.jit.is_scripting()\n            or torch.compiler.is_compiling()\n            or not input.is_cuda\n        ):\n            return manual_rms_norm(input, self.normalized_shape, self.weight, self.eps)\n\n        if self.elementwise_affine:\n            return fused_rms_norm_affine(\n                input,\n                self.weight,\n                self.normalized_shape,\n                self.eps,\n                self.memory_efficient,\n            )\n        else:\n            return fused_rms_norm(input, self.normalized_shape, self.eps, self.memory_efficient)\n\n    def extra_repr(self):\n        return \"{normalized_shape}, eps={eps}, elementwise_affine={elementwise_affine}\".format(\n            **self.__dict__\n        )\n\n\n# NOTE (mkozuki): Why \"mixed\"?\n# MixedFusedLayerNorm differs from FusedLayerNorm in that this layer norm uses parameter's dtype\n# as output tensor's dtype while FusedLayerNorm uses input tensor's dtype for output tensor's dtype.\n# See: `layer_norm_affine` and `layer_norm_affine_mixed_dtypes` in \"csrc/layer_norm_cuda.cpp\"\nclass MixedFusedLayerNorm(FusedLayerNorm):\n    def __init__(self, normalized_shape, eps=1e-5, *, memory_efficient=False, **kwargs):\n        if \"elementwise_affine\" in kwargs:\n            import warnings\n\n            warnings.warn(\"MixedFusedLayerNorm does not support `elementwise_affine` argument\")\n            elementwise_affine = kwargs.pop(\"elementwise_affine\")\n            if not elementwise_affine:\n                raise RuntimeError(\n                    \"MixedFusedLayerNorm does not support `elementwise_affine = False`\"\n                )\n\n        super().__init__(\n            normalized_shape=normalized_shape,\n            eps=eps,\n            elementwise_affine=True,\n            memory_efficient=memory_efficient,\n        )\n\n    def forward(self, input: torch.Tensor):\n        # NOTE (mkozuki): CPU path is here mainly for unittest sake.\n        if (\n            torch.jit.is_tracing()\n            or torch.jit.is_scripting()\n            or torch.compiler.is_compiling()\n            or not input.is_cuda\n        ):\n            return F.layer_norm(input, self.normalized_shape, self.weight, self.bias, self.eps)\n        return mixed_dtype_fused_layer_norm_affine(\n            input,\n            self.weight,\n            self.bias,\n            self.normalized_shape,\n            self.eps,\n            self.memory_efficient,\n        )\n\n\n# MixedFusedLayerNorm differs from FusedLayerNorm in that this layer norm uses parameter's dtype\n# as output tensor's dtype while FusedLayerNorm uses input tensor's dtype for output tensor's dtype.\n# See: `layer_norm_affine` and `layer_norm_affine_mixed_dtypes` in \"csrc/layer_norm_cuda.cpp\"\nclass MixedFusedRMSNorm(FusedRMSNorm):\n    def __init__(self, normalized_shape, eps=1e-5, *, memory_efficient=False, **kwargs):\n        if \"elementwise_affine\" in kwargs:\n            import warnings\n\n            warnings.warn(\"MixedFusedRMSNorm does not support `elementwise_affine` argument\")\n            elementwise_affine = kwargs.pop(\"elementwise_affine\")\n            if not elementwise_affine:\n                raise RuntimeError(\n                    \"MixedFusedRMSNorm does not support `elementwise_affine = False`\"\n                )\n\n        super().__init__(\n            normalized_shape=normalized_shape,\n            eps=eps,\n            elementwise_affine=True,\n            memory_efficient=memory_efficient,\n        )\n\n    def forward(self, input: torch.Tensor):\n        # NOTE (mkozuki): CPU path is here mainly for unittest sake.\n        # TODO Manual RMS Norm Implementation Here\n        if (\n            torch.jit.is_tracing()\n            or torch.jit.is_scripting()\n            or torch.compiler.is_compiling()\n            or not input.is_cuda\n        ):\n            return manual_rms_norm(input, self.normalized_shape, self.weight, self.eps)\n        return mixed_dtype_fused_rms_norm_affine(\n            input, self.weight, self.normalized_shape, self.eps, self.memory_efficient\n        )\n")


In [ ]:
pathlib.Path("probe.py").write_text("\"\"\"Exercise exact Apex setup_context callbacks through real PyTorch custom ops.\n\nForward/backward bodies are identity probes, not Apex normalization kernels.\nThis isolates tensor saving and grad-mode dispatch without compiling extensions.\n\"\"\"\nimport ast\nimport itertools\nimport json\nfrom pathlib import Path\n\nimport torch\n\n\nCALLBACKS = {\n    \"layer_affine\": (\"_fused_layer_norm_affine_setup_context\", 5, 3),\n    \"rms_affine\": (\"_fused_rms_norm_affine_setup_context\", 3, 2),\n    \"layer\": (\"_fused_layer_norm_setup_context\", 3, 3),\n    \"rms\": (\"_fused_rms_norm_setup_context\", 2, 2),\n}\n\n\ndef extract(path, name):\n    tree = ast.parse(Path(path).read_text())\n    matches = [n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef) and n.name == name]\n    assert len(matches) == 1\n    # Execute only the exact callback AST; no imported Apex modules or stand-in ctx.\n    module = ast.fix_missing_locations(ast.Module(body=matches, type_ignores=[]))\n    namespace = {\"torch\": torch}\n    exec(compile(module, path, \"exec\"), namespace)\n    return namespace[name]\n\n\ndef register(version, kind, path):\n    callback_name, expected_saved, outputs = CALLBACKS[kind]\n    callback = extract(path, callback_name)\n    events = []\n    inputs = \"Tensor input, \"\n    if \"affine\" in kind:\n        inputs += \"Tensor weight, \"\n        if kind == \"layer_affine\":\n            inputs += \"Tensor bias, \"\n    inputs += \"SymInt[] normalized_shape, float eps, bool memory_efficient\"\n    schema = f\"({inputs}) -> ({', '.join(['Tensor'] * outputs)})\"\n\n    def forward(*args):\n        x = args[0]\n        auxiliary = torch.zeros(x.shape[0], device=x.device, dtype=torch.float32)\n        return tuple([x.clone()] + [auxiliary.clone() for _ in range(outputs - 1)])\n\n    op = torch.library.custom_op(f\"apex2012probe::{version}_{kind}\", forward,\n                                 mutates_args=(), schema=schema)\n\n    def setup(ctx, inputs, output):\n        events.append({\"grad_enabled\": torch.is_grad_enabled(), \"requires_grad\": inputs[0].requires_grad})\n        callback(ctx, inputs, output)\n\n    def backward(ctx, grad_output, *grad_auxiliary):\n        saved = ctx.saved_tensors\n        if len(saved) != expected_saved:\n            raise RuntimeError(f\"missing saved tensors: expected {expected_saved}, got {len(saved)}\")\n        result = [grad_output]\n        if \"affine\" in kind:\n            result.append(torch.zeros_like(saved[1]))\n            if kind == \"layer_affine\":\n                result.append(torch.zeros_like(saved[2]))\n        return tuple(result + [None, None, None])\n\n    op.register_autograd(backward, setup_context=setup)\n    return op, events\n\n\ndef main():\n    assert torch.cuda.is_available(), \"This evidence run requires CUDA\"\n    print(json.dumps({\"torch\": torch.__version__, \"torch_cuda\": torch.version.cuda,\n                      \"gpu\": torch.cuda.get_device_name(), \"capability\": torch.cuda.get_device_capability()}))\n    records = []\n    for version, path in ((\"base\", \"base.py\"), (\"head\", \"head.py\")):\n        for kind in CALLBACKS:\n            op, events = register(version, kind, path)\n            for device, dtype, memory_efficient, strided in itertools.product(\n                    (\"cpu\", \"cuda\"), (torch.float32, torch.float16), (False, True), (False, True)):\n                x = torch.randn(4, 8, device=device, dtype=dtype)\n                if strided:\n                    x = x.T\n                x.requires_grad_()\n                args = [x]\n                if \"affine\" in kind:\n                    args.append(torch.ones(x.shape[-1], device=device, dtype=dtype, requires_grad=True))\n                    if kind == \"layer_affine\":\n                        args.append(torch.zeros(x.shape[-1], device=device, dtype=dtype, requires_grad=True))\n                args += [[x.shape[-1]], 1e-5, memory_efficient]\n                events.clear()\n                out = op(*args)[0]\n                assert out.requires_grad\n                assert len(events) == 1 and events[0][\"requires_grad\"]\n                callback_grad = events[0][\"grad_enabled\"]\n                error = None\n                try:\n                    out.sum().backward()\n                except RuntimeError as exc:\n                    error = str(exc)\n                if version == \"base\":\n                    assert error is None, error\n                    torch.testing.assert_close(x.grad, torch.ones_like(x))\n                    for parameter in args[1:-3]:\n                        torch.testing.assert_close(parameter.grad, torch.zeros_like(parameter))\n                else:\n                    assert error == f\"missing saved tensors: expected {CALLBACKS[kind][1]}, got 0\", error\n                events.clear()\n                with torch.no_grad():\n                    inference_out = op(*args)[0]\n                assert not inference_out.requires_grad\n                assert len(events) == 0, \"setup_context unexpectedly ran under no_grad\"\n                records.append({\"version\": version, \"kind\": kind, \"device\": device,\n                                \"dtype\": str(dtype), \"memory_efficient\": memory_efficient,\n                                \"strided\": strided, \"callback_grad_enabled\": callback_grad,\n                                \"backward_error\": error, \"no_grad_setup_calls\": len(events)})\n    torch.cuda.synchronize()\n    result = {\"records\": records, \"base_backward_pass\": 64, \"head_backward_missing_saves\": 64,\n              \"no_grad_controls\": 128, \"status\": \"PASS\"}\n    Path(\"probe-results.json\").write_text(json.dumps(result, indent=2))\n    for record in records:\n        print(json.dumps(record))\n    print(json.dumps({k: v for k, v in result.items() if k != \"records\"}))\n\n\nif __name__ == \"__main__\":\n    main()\n")
p=subprocess.run(["python","probe.py"],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(p.stdout)
pathlib.Path("apex2012-probe.log").write_text(p.stdout)
assert p.returncode==0, p.returncode


In [ ]:
print(json.dumps({name:pathlib.Path(name).read_text() for name in ["apex2012-probe.log","probe-results.json"]}))
